# 📺 Ejercicios: YouTube Trending
# USA - 40,000 vídeos reales

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuáles son los vídeos
### más ODIADOS de YouTube
### trending?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/youtube_trending_us.csv'
)

# Un voto = like o dislike
df['votos'] = df['likes'] + df['dislikes']

# % de odio
df['pct_dislikes'] = (
  df['dislikes'] / df['votos'] * 100
)

# Solo videos con +100K votos
odiados = (
  df[df['votos'] > 100_000]
  .drop_duplicates('video_id')
  .nlargest(10, 'pct_dislikes')
  [['title', 'channel_title',
    'pct_dislikes', 'views']]
)

odiados['views_M'] = (
  odiados['views'] / 1e6
)

print("Top 10 más odiados:")
print(
  odiados[['title','pct_dislikes',
           'views_M']]
  .to_string(index=False)
)

### 📊 INTERPRETACIÓN:

**¿Qué estamos midiendo?**
El ratio de odio: qué porcentaje de la gente que vota, vota negativo.

**¿Por qué filtramos por +100K votos?**
Para evitar vídeos pequeños donde 3 dislikes de 5 votos dan un 60% artificial. Queremos vídeos con muestra estadística significativa.

**¿Cómo leer el resultado?**
- `pct_dislikes` > 50% → la MAYORÍA que vota lo hace negativamente. El vídeo genera rechazo activo.
- Si encima tiene millones de views → es odio viral (ej: YouTube Rewind 2018).

**Insight de negocio:**
Un vídeo odiado no es necesariamente malo para el creador. YouTube Rewind 2018 tiene 20M+ dislikes pero 220M views. El odio genera engagement, y el algoritmo premia engagement.

### 💼 PREGUNTA ENTREVISTA:
### ¿El CLICKBAIT funciona?
### ¿Más MAYÚSCULAS = más views? (DONE)

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/youtube_trending_us.csv'
)

bins = [0, 20, 40, 60, 80, 100]
labels = [
  '0-20%', '20-40%',
  '40-60%', '60-80%', '80-100%'
]

# % de letras en mayúsculas
df['caps_pct'] = (
  df['title'].str.count('[A-Z]')
  / df['title'].str.len() * 100
)

df['caps_bin'] = pd.cut(
  df['caps_pct'],
  bins=bins, labels=labels
)

result = (
  df
  .groupby('caps_bin',
    observed=False)
  ['views']
  .agg(['median', 'count'])
)

result['median_M'] = (
  result['median'] / 1e6
)

print("Views por % mayúsculas:")
print(result[['median_M','count']]
  .round(2))

### 📊 INTERPRETACIÓN:

**¿Qué estamos midiendo?**
Correlación entre agresividad del título (% de mayúsculas) y views.

**¿Cómo leer la tabla?**
- Cada fila es un rango de mayúsculas (0-20% es título normal, 80-100% es "TODO GRITANDO").
- `median_M` = mediana de views en millones para ese grupo.
- `count` = cuántos vídeos caen en ese rango (para saber si la muestra es fiable).

**¿Qué dice el boost?**
Si sale +40% significa que los títulos con 60-80% mayúsculas tienen un 40% más de views medianas que los títulos normales (0-20%).

**Cuidado con la causalidad:**
Correlación ≠ causa. Los vídeos con clickbait pueden tener más views porque los canales grandes usan clickbait, no porque las mayúsculas atraigan. Pero el dato está ahí: funciona como señal.

**Insight para creadores:**
Hay un sweet spot (40-80%). Título 100% mayúsculas puede parecer spam y ser penalizado por CTR bajo.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué canales MONOPOLIZAN
### el trending de YouTube?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/youtube_trending_us.csv'
)

# Apariciones en trending
top = (
  df
  .groupby('channel_title')
  .agg(
    dias_trending=(
      'trending_date','nunique'),
    videos=(
      'video_id','nunique'),
    views_M=('views','mean')
  )
  .sort_values('dias_trending',
    ascending=False)
  .head(15)
)

top['views_M'] = top['views_M'] / 1e6

# % trending acaparado por top 15
total_dias = (
  df['trending_date'].nunique()
)
monopolio = (
  top['dias_trending'].mean()
  / total_dias * 100
)

print("Top 15 canales trending:")
print(top.round(1))
print(f"\nPresentes {monopolio:.0f}%")
print("de los días del año")

### 📊 INTERPRETACIÓN:

**¿Qué estamos midiendo?**
Concentración del trending: si unos pocos canales acaparan la mayoría de apariciones.

**¿Cómo leer la tabla?**
- `dias_trending` = en cuántos días distintos ese canal tuvo al menos un vídeo en trending.
- `videos` = cuántos vídeos distintos del canal aparecieron en trending.
- `views_M` = media de views en millones de sus vídeos trending.

**¿Qué significa el % final?**
Si dice "Presentes 65% de los días del año" → el top 15 de canales aparece en trending 2 de cada 3 días. El trending NO es democrático.

**Insight de negocio:**
YouTube trending es un oligopolio. Los canales con presencia constante (deportes, late shows, música) dominan porque publican a diario. Un canal pequeño puede llegar a trending pero casi nunca repite. Es el "rich get richer" del algoritmo.

### 💼 PREGUNTA ENTREVISTA:
### ¿El SPAM de tags funciona?
### ¿Más tags = más views?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/youtube_trending_us.csv'
)

# Contar tags (separados por |)
df['n_tags'] = (
  df['tags'].str.count('\\|') + 1
)
df.loc[
  df['tags']=='[none]', 'n_tags'
] = 0

bins = [0, 1, 5, 10, 20, 40, 200]
labels = [
  '0', '1-5', '5-10',
  '10-20', '20-40', '40+'
]

df['tag_bin'] = pd.cut(
  df['n_tags'],
  bins=bins, labels=labels,
  right=False
)

result = (
  df
  .groupby('tag_bin',
    observed=False)
  .agg(
    views_mediana=('views','median'),
    n_videos=('views','count')
  )
)

result['views_mediana'] = (
  result['views_mediana'] / 1e6
)

print("Views por nº de tags:")
print(result.round(2))

boost = (
  result.loc['20-40',
    'views_mediana']
  / result.loc['1-5',
    'views_mediana']
  - 1
) * 100
print(f"\n+{boost:.0f}% con 20-40 tags")
print("vs solo 1-5 tags")

### 📊 INTERPRETACIÓN:

**¿Qué estamos midiendo?**
Si meter más tags en un vídeo está asociado a más views.

**¿Cómo leer la tabla?**
- Cada fila es un rango de número de tags (0 tags, 1-5, 5-10, etc.).
- `views_mediana` = mediana de views (en millones) de los vídeos en ese rango.
- Usamos mediana y no media para evitar que un vídeo viral distorsione el grupo.

**¿Qué dice el boost?**
Si sale "+80% con 20-40 tags vs 1-5" → los vídeos con muchos tags tienen el doble de views medianas.

**¿Significa que debo meter 40 tags?**
No necesariamente. Cuidado con el sesgo de supervivencia: los vídeos trending ya son exitosos, y los canales grandes usan más tags porque tienen equipos de SEO. Pero el patrón es claro: dentro de trending, más tags correlaciona con más views.

**Insight para creadores:**
El sweet spot suele estar en 15-30 tags relevantes. 0 tags es tirarte un tiro en el pie. 40+ no penaliza, pero los retornos son decrecientes.

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuántos días tarda un
### vídeo en hacerse trending
### y qué % lo logra en 24h?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/youtube_trending_us.csv'
)

df['publish_time'] = pd.to_datetime(
  df['publish_time'], utc=True
).dt.tz_localize(None)

# trending_date: formato YY.DD.MM
df['trending_dt'] = pd.to_datetime(
  df['trending_date'],
  format='%y.%d.%m'
)

df['dias'] = (
  (df['trending_dt']
   - df['publish_time']).dt.days
)

valid = df[df['dias'] >= 0]

rapido = (
  (valid['dias'] <= 1).mean() * 100
)

print("Días hasta trending:")
print(f"Mediana: {valid['dias'].median():.0f}")
print(f"Media: {valid['dias'].mean():.1f}")
print()
print(f"Solo {rapido:.0f}% llega")
print("en menos de 24h")
print()

# Los que tardan +7 dias?
lentos = (
  (valid['dias'] > 7).mean() * 100
)
print(f"{lentos:.0f}% tarda +1 semana")

### 📊 INTERPRETACIÓN:

**¿Qué estamos midiendo?**
El tiempo que pasa desde que se publica un vídeo hasta que aparece en trending por primera vez.

**¿Cómo leer los resultados?**
- `Mediana` = el valor que divide la distribución por la mitad. Si es 2, la mitad de los vídeos llegan a trending en 2 días o menos.
- `Media` > mediana → hay una cola larga de vídeos que tardan mucho (distribución asimétrica).
- El % que llega en <24h te dice cuántos son "virales instantáneos".

**¿Por qué filtramos `dias >= 0`?**
Hay datos sucios donde trending_date es anterior a publish_time (errores de timezone o formato). Los descartamos.

**Insight de negocio:**
Si solo un ~30% llega en 24h, el trending NO es solo contenido fresco. YouTube también sube a trending vídeos de hace días/semanas que están ganando tracción. Esto significa que tu vídeo tiene una ventana de oportunidad más larga de lo que crees: no todo se juega en las primeras horas.